In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Silver to Gold — Build Star Schema

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import LongType
from pyspark.sql.window import Window

storage_key = dbutils.secrets.get(scope="retailrocket", key="storage-key")
spark.conf.set("fs.azure.account.key.stdportfolio.blob.core.windows.net", storage_key)

# COMMAND ----------

# MAGIC %md
# MAGIC ### 0. Determine Target Schema (dev vs prod)

# COMMAND ----------

# Target schema from job parameter, default to prod schema 
try:
    target_schema = dbutils.widgets.get("target_schema")
except Exception:
    target_schema = "dbo"

print(f"Target Azure SQL schema: {target_schema}")

# COMMAND ----------

# Drop existing gold tables (schema changed)
spark.sql("DROP TABLE IF EXISTS gold.dim_date")
spark.sql("DROP TABLE IF EXISTS gold.dim_users")
spark.sql("DROP TABLE IF EXISTS gold.dim_categories")
spark.sql("DROP TABLE IF EXISTS gold.dim_items")
spark.sql("DROP TABLE IF EXISTS gold.dim_event_type")
spark.sql("DROP TABLE IF EXISTS gold.fact_events")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1. Load Silver Tables

# COMMAND ----------

events = spark.table("silver.events")
category_tree = spark.table("silver.category_tree")
item_properties = spark.table("silver.item_properties")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2. Build Dimension Tables

# COMMAND ----------

# MAGIC %md
# MAGIC #### 2.1 dim_date

# COMMAND ----------

dim_date = (
    events
    .select(F.to_date("event_timestamp").alias("full_date"))
    .distinct()
    .withColumn("date_sk", F.monotonically_increasing_id())
    .withColumn("year", F.year("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))  # 1=Sun, 7=Sat
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .cache()  # Prevent different IDs generated when joining with fact_events.
)

print(f"dim_date: {dim_date.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC #### 2.2 dim_users

# COMMAND ----------

dim_users = (
    events
    .groupBy("visitorid")
    .agg(
        F.min("event_timestamp").alias("first_event_ts"),
        F.max("event_timestamp").alias("last_event_ts"),
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("event") == "transaction", 1).otherwise(0)).alias("total_purchases")
    )
    .withColumn("has_purchased", F.col("total_purchases") > 0)
    .withColumn("user_sk", F.monotonically_increasing_id())
    .cache()  # Prevent different IDs generated when joining with fact_events.
)

print(f"dim_users: {dim_users.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC #### 2.3 dim_categories

# COMMAND ----------

dim_categories = (
    category_tree
    .withColumn("category_sk", F.monotonically_increasing_id())
    .withColumnRenamed("categoryid", "category_id")
    .withColumnRenamed("parentid", "parent_id")
    .cache()
)

print(f"dim_categories: {dim_categories.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC #### 2.4 dim_items (FIXED — all items from events, left join categories)

# COMMAND ----------

# Get latest category per item
category_props = item_properties.filter(F.col("property") == "categoryid")

latest_category = (
    category_props
    .withColumn("row_number", F.row_number().over(
        Window.partitionBy("itemid").orderBy(F.desc("timestamp_ms"))
    ))
    .filter(F.col("row_number") == 1)
    .select("itemid", F.col("value").cast("long").alias("categoryid"))
)

# Build dim_items from ALL items in events
all_items = events.select("itemid").distinct()

dim_items = (
    all_items
    .withColumn("item_sk", F.monotonically_increasing_id())
    .join(latest_category, "itemid", "left")
    .join(dim_categories, F.col("categoryid") == dim_categories["category_id"], "left")
    .select(
        "item_sk",
        "itemid",
        "category_sk"  # NULL for items without categories
    )
    .cache()  # Prevent different IDs generated when joining with fact_events.
)

print(f"dim_items: {dim_items.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC #### 2.5 dim_event_type

# COMMAND ----------

dim_event_type = (
    events
    .select("event")
    .distinct()
    .withColumn("event_type_sk", F.monotonically_increasing_id())
    .withColumnRenamed("event", "event_type")
    .cache()  # Prevent different IDs generated when joining with fact_events.
)

print(f"dim_event_type: {dim_event_type.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 3. Build Fact Table

# COMMAND ----------

fact_events = (
    events
    .join(dim_date, events["event_timestamp"].cast("date") == dim_date["full_date"], "left")
    .join(dim_users, "visitorid", "left")
    .join(dim_items, "itemid", "left")
    .join(dim_event_type, events["event"] == dim_event_type["event_type"], "left")
    .select(
        F.monotonically_increasing_id().alias("event_sk"),
        "date_sk",
        "user_sk",
        "item_sk",
        "event_type_sk",
        "transactionid"
    )
)

print(f"fact_events: {fact_events.count():,} rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4. Write Gold Delta Tables

# COMMAND ----------

dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")
dim_users.write.format("delta").mode("overwrite").saveAsTable("gold.dim_users")
dim_categories.write.format("delta").mode("overwrite").saveAsTable("gold.dim_categories")
dim_items.write.format("delta").mode("overwrite").saveAsTable("gold.dim_items")
dim_event_type.write.format("delta").mode("overwrite").saveAsTable("gold.dim_event_type")
fact_events.write.format("delta").mode("overwrite").saveAsTable("gold.fact_events")

print("Gold tables written.")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5. Load to Azure SQL (JDBC) — schema-aware

# COMMAND ----------

jdbc_url = (
    "jdbc:sqlserver://dw-retailrocket1.database.windows.net:1433;"
    "database=retailrocket-dw;"
    "connectRetryCount=3;"        # retry 3 times
    "connectRetryInterval=10;"    # wait 10 seconds between retries
    "connectionTimeout=30;"       # 30 second timeout
)
jdbc_properties = {
    "user": "etl_loader",
    "password": dbutils.secrets.get(scope="retailrocket", key="sql-password"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

import time

# Qualified table name: <schema>.<table>
def qualified(table):
    return f"{target_schema}.{table}"

def write_with_retry(df, table, url, props, max_retries=3):
    for attempt in range(max_retries):
        try:
            df.write.jdbc(url, qualified(table), mode="overwrite", properties=props)
            print(f"  {qualified(table)}: written")
            return
        except Exception as e:
            print(f"  {qualified(table)}: attempt {attempt+1} failed — {e}")
            if attempt < max_retries - 1:
                time.sleep(10)
            else:
                raise

write_with_retry(dim_date, "dim_date", jdbc_url, jdbc_properties)
write_with_retry(dim_users, "dim_users", jdbc_url, jdbc_properties)
write_with_retry(dim_categories, "dim_categories", jdbc_url, jdbc_properties)
write_with_retry(dim_items, "dim_items", jdbc_url, jdbc_properties)
write_with_retry(dim_event_type, "dim_event_type", jdbc_url, jdbc_properties)
write_with_retry(fact_events, "fact_events", jdbc_url, jdbc_properties)

print(f"Azure SQL tables loaded into schema '{target_schema}'.")